In [1]:
import pandas as pd

In [ ]:

def airgo_robust_load(filepath):

    data = pd.read_csv(filepath, nrows=5)

    # FORMAT MANUAL DOWNLOAD:
    if data.iloc[0,0] == 'date':

        data= pd.read_csv(filepath, skiprows = 1)
        data.columns = data.columns.str.strip()
        data.date = data.date.apply(lambda x: x+' ')
        data['DateTime'] = pd.to_datetime(data.date+data.time, infer_datetime_format = 1)
        data.drop('date', axis = 1, inplace = True)
        data.drop('time', axis = 1, inplace = True)

        data = data[['DateTime', 'accX' , 'accY',  'accZ',  'Band', 'CRCStatus']]
        bad_crc_index = data.index[np.logical_not(pd.isna(data.CRCStatus))]
        data.drop(bad_crc_index, inplace=True)
        data.drop('CRCStatus', axis=1, inplace=True)
        data.columns = ['datetime', 'accx', 'accy','accz','band']

    # FORMAT SERVER DOWNLOAD:
    else:
        data = pd.read_csv(filepath, names=['datetime', 'accx', 'accy','accz','band'])
        data.datetime = pd.to_datetime(data.datetime, infer_datetime_format=1)

    return data

In [ ]:
data = airgo_robust_load('AirGo_7ac453acddfb_2020-04-27_20_49_57_raw.csv')
data

In [ ]:
g = 9.81
fs=10
for acc in ['accx','accy', 'accz']:
    data[acc] /=g
    data[acc+'_1sec'] = data[acc].reset_index()[acc].rolling(fs, center=True).mean().values
    data[acc+'_var_1sec'] = data[acc].reset_index()[acc].rolling(fs, center=True).var().values
    data[acc+'_var_10sec'] = data[acc].reset_index()[acc].rolling(10*fs, center=True).var().values

In [ ]:
data['activity_index_1sec'] = np.sqrt(np.maximum([0],np.mean([data['accx_var_1sec'], data['accy_var_1sec'], data['accz_var_1sec']], axis=0)))
data['activity_index_10sec'] = np.sqrt(np.maximum([0],np.mean([data['accx_var_10sec'], data['accy_var_10sec'], data['accz_var_10sec']], axis=0)))
data['acc_norm'] = np.maximum([0],np.sqrt(data.accx**2+data.accy**2+data.accz**2))
data['acc_norm']

In [ ]:
data['acc_norm'] = np.maximum([0],np.sqrt(data.accx**2+data.accy**2+data.accz**2)-0.98)
data['acc_norm'] = data['acc_norm'].reset_index()['acc_norm'].rolling(fs, center=True).mean().values
data['acc_norm_10sec'] = data['acc_norm'].reset_index()['acc_norm'].rolling(10*fs, center=True).mean().values

In [ ]:
peaks = find_peaks(data.activity_index_10sec, prominence=0.2)[0]
peaks = np.concatenate([[0], peaks])
peaks_device = peaks[::5]
peaks_physio = np.array(list(set(peaks) - set(peaks_device)))
peaks_physio.sort()

In [ ]:
data_sel = data.iloc[:120*60*fs]
plt.figure(figsize=(12,6))
ax1 = plt.subplot(311)
plt.plot(data_sel.accx,label='x')
plt.plot(data_sel.accy,label='y')
plt.plot(data_sel.accz,label='z')
plt.legend(ncol=3)
plt.subplot(312, sharex=ax1)
plt.plot(data_sel.activity_index_1sec,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.activity_index_10sec,c='r', zorder=2)
plt.scatter(peaks, data_sel.activity_index_10sec[peaks],c='k')
plt.scatter(peaks_device, data_sel.activity_index_10sec[peaks_device],c='b')

plt.legend(['1sec average', '10sec average', 'physiological', 'device'])

plt.ylabel('Activity Index (g)')
plt.subplot(313, sharex=ax1)
plt.plot(data_sel.acc_norm,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.acc_norm_10sec,c='r', zorder=2)
plt.ylabel('ENMO (g)')
plt.legend(['1sec average', '10sec average'])
plt.savefig('omar_recording_actigraphy.png', dpi=200)
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 8})

fig = plt.figure(figsize=(12,8))
for i, peak_idx in enumerate(peaks_physio):
    fig.add_subplot(5,4,i+1)
    plt.plot(data.band[peak_idx-30*fs: peak_idx+30*fs])
for i, peak_idx in enumerate(peaks_device[1:]):
    fig.add_subplot(5,4,i+17)
    plt.plot(data.band[peak_idx-30*fs: peak_idx+30*fs], c='blue')
    
plt.suptitle('physiological / position change (prone) vs. device position change (no proning)')
plt.savefig('breathing_prone_vs_non_60sec.png',dpi=200)

    


In [ ]:
plt.rcParams.update({'font.size': 8})

fig = plt.figure(figsize=(12,8))
for i, peak_idx in enumerate(peaks_physio):
    fig.add_subplot(5,4,i+1)
    plt.plot(data.band[peak_idx-15*fs: peak_idx+15*fs])
for i, peak_idx in enumerate(peaks_device[1:]):
    fig.add_subplot(5,4,i+17)
    plt.plot(data.band[peak_idx-15*fs: peak_idx+15*fs], c='blue')
    
plt.suptitle('physiological / position change (prone) vs. device position change (no proning)')
plt.savefig('breathing_prone_vs_non_30sec.png',dpi=200)
    


In [ ]:
from sklearn.cluster import KMeans
import numpy as np

X = data[['accx_1sec','accy_1sec','accz_1sec']].dropna().values

kmeans = KMeans(n_clusters=4, random_state=0).fit(X)
kmeans.labels_

data['position_cluster'] = -1
data.loc[data[['accx_1sec','accy_1sec','accz_1sec']].dropna().index, 'position_cluster'] = kmeans.labels_

In [ ]:
from sklearn.cluster import KMeans

# X = data[['accx_1sec','accy_1sec','accz_1sec']].dropna().values

def acceleration_position(data):
    data['position_cluster'] = -1
#     cluster_names = ['right_lateral','left_lateral','suspine','prone']
    cluster_centers = np.array([
        [-0.10528739, -0.90177368,  0.07899741],
        [-0.14248938,  0.96973593, -0.06413331],
        [-0.16258391, -0.05833847, -0.9190312 ],
        [ 0.01891357,  0.09631911,  0.98882481]
    ])

    kmeans = KMeans(n_clusters=4, init=cluster_centers)
    kmeans.cluster_centers_ = cluster_centers
    index_valid = data[['accx_1sec','accy_1sec','accz_1sec']].dropna().index
    data.loc[index_valid, 'position_cluster'] = kmeans.predict(data[['accx_1sec','accy_1sec','accz_1sec']].dropna())
    
    return data



In [ ]:
cluster_centers = np.array([
    [-0.10528739, -0.90177368,  0.07899741],
    [-0.14248938,  0.96973593, -0.06413331],
    [-0.16258391, -0.05833847, -0.9190312 ],
    [ 0.01891357,  0.09631911,  0.98882481]
])

kmeans = KMeans(n_clusters=4, init=cluster_centers)

In [ ]:
data2 = data.copy()
data = acceleration_position(data)

In [ ]:
colors = [(0.6,0.6,0.3), (0.3,0.4,0.4), (0.8, 0.8,0.1), (0.5, 0.2, 0.7)]

colors = ['r','g','b','k']
data_sel = data.iloc[:120*60*fs]
plt.figure(figsize=(12,6))
ax1 = plt.subplot(311)
plt.plot(data_sel.accx,label='x')
plt.plot(data_sel.accy,label='y')
plt.plot(data_sel.accz,label='z')
for i in range(4):
    idx_cluster = data.position_cluster[data.position_cluster==i].index
    tmp = np.empty(data.position_cluster.shape)
    tmp[:] = np.nan
    tmp[idx_cluster] = 2
    plt.plot(data.index, tmp, c=colors[i])

plt.legend(ncol=3)
plt.subplot(312, sharex=ax1)
plt.plot(data_sel.activity_index_1sec,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.activity_index_10sec,c='r', zorder=2)
plt.scatter(peaks, data_sel.activity_index_10sec[peaks],c='k')
plt.scatter(peaks_device, data_sel.activity_index_10sec[peaks_device],c='b')

plt.legend(['1sec average', '10sec average', 'physiological', 'device'])

plt.ylabel('Activity Index (g)')
plt.subplot(313, sharex=ax1)
plt.plot(data_sel.acc_norm,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.acc_norm_10sec,c='r', zorder=2)
plt.ylabel('ENMO (g)')
plt.legend(['1sec average', '10sec average'])
plt.savefig('omar_recording_actigraphy.png', dpi=200)
plt.show()

In [ ]:
data_sel = data.iloc[:120*60*fs]
plt.figure(figsize=(12,6))
ax1 = plt.subplot(211)
plt.plot(data_sel.accx,label='x')
plt.plot(data_sel.accy,label='y')
plt.plot(data_sel.accz,label='z')
plt.legend(ncol=3)


In [ ]:
peaks = find_peaks(data.activity_index_10sec, prominence=0.2)[0]
peaks = np.concatenate([[0], peaks])
peaks_device = peaks[::5]
peaks_physio = np.array(list(set(peaks) - set(peaks_device)))
peaks_physio.sort()

In [ ]:
plt.rcParams.update({'font.size': 8})

fig = plt.figure(figsize=(12,8))
for i, peak_idx in enumerate(peaks_physio):
    fig.add_subplot(5,4,i+1)
    plt.plot(data.band[peak_idx-30*fs: peak_idx+30*fs])
for i, peak_idx in enumerate(peaks_device[1:]):
    fig.add_subplot(5,4,i+17)
    plt.plot(data.band[peak_idx-30*fs: peak_idx+30*fs], c='blue')
    
plt.suptitle('physiological / position change (prone) vs. device position change (no proning)')
plt.savefig('breathing_prone_vs_non_60sec.png',dpi=200)

    


In [ ]:
data_sel = data.iloc[:120*60*fs]
plt.figure(figsize=(12,6))
ax1 = plt.subplot(311)
plt.plot(data_sel.accx,label='x')
plt.plot(data_sel.accy,label='y')
plt.plot(data_sel.accz,label='z')
plt.legend(ncol=3)
plt.subplot(312, sharex=ax1)
plt.plot(data_sel.activity_index_1sec,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.activity_index_10sec,c='r', zorder=2)
plt.scatter(peaks, data_sel.activity_index_10sec[peaks],c='k')
plt.scatter(peaks_device, data_sel.activity_index_10sec[peaks_device],c='b')

plt.legend(['1sec average', '10sec average', 'physiological', 'device'])

plt.ylabel('Activity Index (g)')
plt.subplot(313, sharex=ax1)
plt.plot(data_sel.acc_norm,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.acc_norm_10sec,c='r', zorder=2)
plt.ylabel('ENMO (g)')
plt.legend(['1sec average', '10sec average'])
plt.savefig('omar_recording_actigraphy.png', dpi=200)
plt.show()

In [ ]:
data_sel = data # data.iloc[:120*60*fs]


colors = ['r','g','b','k']
#     cluster_names = ['right_lateral','left_lateral','suspine','prone']

plt.figure(figsize=(12,6))
ax1 = plt.subplot(311)
plt.plot(data_sel.accx,label='x')
plt.plot(data_sel.accy,label='y')
plt.plot(data_sel.accz,label='z')

for i in range(4):
#     idx_cluster = data.position_cluster[data.position_cluster==i].index
    idx_cluster = np.where(data.position_cluster==i)[0]

    tmp = np.empty(data.position_cluster.shape)
    tmp[:] = np.nan
    tmp[idx_cluster] = 1.5
    plt.plot(data.index, tmp, c=colors[i])
    
    
plt.legend(ncol=3)
plt.subplot(312, sharex=ax1)
plt.plot(data_sel.acc_ai_1sec,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.acc_ai_10sec,c='r', zorder=2)

plt.legend(['1sec average', '10sec average', 'physiological', 'device'])

plt.ylabel('Activity Index (g)')
plt.subplot(313, sharex=ax1)
plt.plot(data_sel.acc_enmo_1sec,c='orange', alpha=0.7, zorder=1)
plt.plot(data_sel.acc_enmo_10sec,c='r', zorder=2)
plt.ylabel('ENMO (g)')
plt.legend(['1sec average', '10sec average'])
plt.savefig('omar_recording_actigraphy.png', dpi=200)
plt.show()